### Analogy Analysis

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import matplotlib.pyplot as plt

In [ ]:
MODEL_NAME = "google/gemma-2-2b-it"
MAX_NEW_TOKENS = 20
OUTPUT_FILE = "gender_analogies_data_output.csv"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)

model.eval()

In [ ]:
df = pd.read_csv(
    "gender-analogies.txt",
    header=None,
    names=["A", "B", "C", "Expected", "Category"]
)

df["Prompt"] = df.apply(
    lambda row: f"""
Complete the analogy.

{row.A} is to {row.B} as {row.C} is to ____.

Respond with ONE word only.
Answer:
""",
    axis=1
)

In [ ]:
def get_answer(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return text[len(prompt):].strip()

In [ ]:
answers = []

for prompt in df["Prompt"]:
    answers.append(get_answer(prompt))

df["Model_Output"] = answers

In [ ]:
import re

def clean_output(text):
    word = re.findall(r"[a-zA-Z]+", text)
    return word[0].lower() if word else ""

In [ ]:
df["Model_Output"] = [clean_output(get_answer(p)) for p in df["Prompt"]]

df["Correct"] = (
    df["Model_Output"]
    .str.lower()
    .str.strip()
    == df["Expected"].str.lower()
)

In [ ]:
accuracy = df.groupby("Dataset")["Correct"].mean()

accuracy

In [ ]:
plt.figure(figsize=(6,4))

accuracy.plot(kind="bar")

plt.ylabel("Accuracy")
plt.title("Analogy Accuracy by Dataset")
plt.ylim(0,1)

plt.xticks(rotation=0)

plt.tight_layout()